# ML baselines on the diplotype-grouped 10-fold CV

Runs Random Forest, SVM (poly) and XGBoost on **exactly the same** diplotype-disjoint
outer folds used by `07_GenotypeOnlyCNN_DiplotypeGroupedCV` and
`Multimodal...-`, so every row of Table 1 comes from one protocol.

What this notebook fixes relative to `BMC머신러닝 성능비교`:

1. **Same folds.** Loads `diplotype_grouped_10fold_assignments.csv`
   (regenerates it identically if missing) instead of the `train_452/test_158`
   sample-level split that Reviewers 1 and 3 flagged as contaminated.
2. **No SVD leakage.** `TruncatedSVD` is fit on the outer-train rows of each fold
   only. The previous notebook fit it on the whole training set once.
3. **Hyperparameters are actually used.** The previous notebook ran
   `RandomizedSearchCV` and then hard-coded different values from an earlier run.
   Here the search result is what gets fit.
4. **Group-aware inner CV.** Hyperparameter search inside each fold also holds out
   whole diplotypes, so no diplotype reaches a model through tuning either.
5. **Per-fold metrics.** Produces 10 values per model, so Table 1 can carry
   mean ± SD and paired p-values on every row (Reviewer 2, point 4).

Outputs: `ml_baselines_diplotype_grouped_fold_metrics.csv`,
`ml_baselines_diplotype_grouped_oof_predictions.csv`,
`table1_grouped_cv_summary.csv`, `table1_grouped_cv_paired_tests.csv`.


## 1. Setup

In [1]:
import os, warnings, json
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, log_loss)
from scipy.stats import randint, uniform, loguniform, ttest_rel, wilcoxon

from xgboost import XGBClassifier
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

warnings.filterwarnings('ignore')

# ---- config (kept identical to the deep-model notebooks where it matters) ----
SEED        = 42
N_SPLITS    = 5          # outer folds
INNER_SPLITS= 3           # group-aware inner CV for hyperparameter search
N_CLASSES   = 3

DATA_CSV      = 'dataset.csv'
FOLD_CSV      = 'diplotype_grouped_5fold_assignments.csv'
ONEHOT_FOLDER = 'onehot_7'

TARGET_ROWS = 4330
TARGET_COLS = 7
ECFP_BITS   = 1024
ECFP_RADIUS = 2
SVD_MAX_COMPONENTS = 500

# SEARCH=True  -> RandomizedSearchCV inside every outer fold (slow, correct)
# SEARCH=False -> fixed params below (fast smoke test; do NOT report these)
SEARCH = True
N_ITER  = 20

np.random.seed(SEED)
print('config ok')

config ok


## 2. Data and diplotype groups

Same `canonical_diplotype` rule as the deep notebooks: `A; B` and `B; A` are one group.

In [2]:
data = pd.read_csv(DATA_CSV).copy()
assert {'Variant','Class','Drug','isomeric'}.issubset(data.columns)

def canonical_diplotype(v):
    alleles = [x.strip() for x in str(v).split(';') if x.strip()]
    return '; '.join(sorted(alleles))

data['DiplotypeGroup'] = data['Variant'].map(canonical_diplotype)

label_encoder = LabelEncoder()
data['ClassEncoded'] = label_encoder.fit_transform(data['Class'])
class_mapping = {i: c for i, c in enumerate(label_encoder.classes_)}

y      = data['ClassEncoded'].values
groups = data['DiplotypeGroup'].values

print('shape              :', data.shape)
print('diplotype groups   :', data['DiplotypeGroup'].nunique())
print('drugs              :', data['Drug'].nunique())
print('class mapping      :', class_mapping)
print(data['Class'].value_counts().to_dict())

shape              : (622, 12)
diplotype groups   : 104
drugs              : 59
class mapping      : {0: 'de', 1: 'f', 2: 'nor'}
{'de': 250, 'f': 195, 'nor': 177}


## 3. Load (or reproduce) the shared fold assignment

If `diplotype_grouped_10fold_assignments.csv` is present it is used verbatim, so the ML rows sit on literally the same folds as the deep rows. Otherwise it is regenerated with the same seed and settings, and the result is checked for diplotype leakage.

In [3]:
def build_folds():
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fid = np.full(len(data), -1, dtype=int)
    for k, (_, te) in enumerate(cv.split(np.zeros(len(y)), y, groups)):
        fid[te] = k + 1
    return fid

if os.path.exists(FOLD_CSV):
    fa = pd.read_csv(FOLD_CSV)
    assert len(fa) == len(data), f'{FOLD_CSV} has {len(fa)} rows, data has {len(data)}'
    fold_id = fa['fold'].values.astype(int)
    print(f'Loaded fold assignment from {FOLD_CSV}')
    regen = build_folds()
    print('matches a fresh regeneration:', bool((regen == fold_id).all()))
else:
    fold_id = build_folds()
    out = data[['Variant','DiplotypeGroup','Drug','Class']].copy()
    out['fold'] = fold_id
    out.to_csv(FOLD_CSV, index=False)
    print(f'{FOLD_CSV} not found -> regenerated and saved')

assert (fold_id > 0).all()

for k in range(1, N_SPLITS + 1):
    te = np.where(fold_id == k)[0]
    tr = np.where(fold_id != k)[0]
    overlap = set(groups[tr]) & set(groups[te])
    assert not overlap, f'fold {k} diplotype leakage: {overlap}'
    cnt = np.bincount(y[te], minlength=N_CLASSES)
    print(f'fold {k:2d}: train={len(tr):3d} test={len(te):3d} '
          f'test groups={len(set(groups[te])):2d} class counts={cnt.tolist()}')

Loaded fold assignment from diplotype_grouped_5fold_assignments.csv
matches a fresh regeneration: True
fold  1: train=530 test= 92 test groups=20 class counts=[38, 31, 23]
fold  2: train=542 test= 80 test groups=16 class counts=[59, 15, 6]
fold  3: train=392 test=230 test groups=24 class counts=[65, 80, 85]
fold  4: train=490 test=132 test groups=23 class counts=[37, 45, 50]
fold  5: train=534 test= 88 test groups=21 class counts=[51, 24, 13]


## 4. Raw features

Genotype: the two allele one-hot matrices are **summed** into (4330, 7) and flattened, matching the existing ML baseline (the deep models stack them into (8660, 7) instead — worth one sentence in Methods).

Drug: 1024-bit Morgan/ECFP4 fingerprint, the same drug information the GCN receives.

SVD is deliberately **not** applied here; it is fit per fold in section 6.

In [4]:
onehot = {}
bad_files = []

for f in sorted(os.listdir(ONEHOT_FOLDER)):
    if not f.endswith('.csv'):
        continue
    key = f.split('.csv')[0].replace('_', '.')
    raw = pd.read_csv(os.path.join(ONEHOT_FOLDER, f), header=None)

    # Diagnose before cleaning: non-numeric cells become NaN on coercion.
    num = raw.apply(pd.to_numeric, errors='coerce')
    m = num.values.astype(np.float64)
    n_nan = int(np.isnan(m).sum())
    n_inf = int(np.isinf(m).sum())
    if n_nan or n_inf:
        bad_files.append({'file': f, 'shape': m.shape, 'n_nan': n_nan, 'n_inf': n_inf})

    m = np.nan_to_num(m, nan=0.0, posinf=0.0, neginf=0.0)
    pad = np.zeros((TARGET_ROWS, TARGET_COLS), dtype=np.float64)
    r, c = min(m.shape[0], TARGET_ROWS), min(m.shape[1], TARGET_COLS)
    pad[:r, :c] = m[:r, :c]
    onehot[key] = pad

print('one-hot matrices loaded:', len(onehot))
if bad_files:
    bf = pd.DataFrame(bad_files)
    print(f'\n*** {len(bf)} one-hot file(s) contained NaN/Inf and were zero-filled ***')
    print('These same files feed the CNN and multimodal notebooks, so check them there too.')
    display(bf)
    bf.to_csv('onehot_files_with_nonfinite_values.csv', index=False)
else:
    print('all one-hot files are finite')

def variant_to_sum(v):
    alleles = [x.strip() for x in str(v).split(';') if x.strip()][:2]
    mats = []
    for a in alleles:
        if a not in onehot:
            raise KeyError(f'one-hot matrix missing for {a}')
        mats.append(onehot[a])
    while len(mats) < 2:
        mats.append(np.zeros((TARGET_ROWS, TARGET_COLS), dtype=np.float64))
    return mats[0] + mats[1]

X_gen = np.stack([variant_to_sum(v) for v in data['Variant']]).reshape(len(data), -1)
print('genotype features :', X_gen.shape, X_gen.dtype,
      '| min', X_gen.min(), '| max', X_gen.max())

def smiles_to_ecfp(s):
    mol = Chem.MolFromSmiles(str(s))
    arr = np.zeros((ECFP_BITS,), dtype=np.float64)
    if mol is None:
        print('[WARN] invalid SMILES:', s)
        return arr
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=ECFP_RADIUS, nBits=ECFP_BITS)
    tmp = np.zeros((ECFP_BITS,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, tmp)
    return tmp.astype(np.float64)

X_drug = np.vstack([smiles_to_ecfp(s) for s in data['isomeric']])
print('drug features     :', X_drug.shape, '| min', X_drug.min(), '| max', X_drug.max())

# Hard guarantee for everything downstream.
assert np.isfinite(X_gen).all(),  'non-finite values remain in X_gen'
assert np.isfinite(X_drug).all(), 'non-finite values remain in X_drug'
print('\nfeature matrices are finite')


one-hot matrices loaded: 361
all one-hot files are finite
genotype features : (622, 30310) float64 | min 0.0 | max 2.0
drug features     : (622, 1024) | min 0.0 | max 1.0

feature matrices are finite


## 5. Metrics

Identical to `compute_metrics` in the deep notebooks so the rows are directly comparable. A class absent from a fold's test set yields NaN for that class's AUROC/AUPR and is skipped by the macro average; folds where AUROC is undefined are reported as NaN and excluded (with n stated) rather than silently dropped.

In [5]:
def compute_metrics(y_true, probs, n_classes=N_CLASSES):
    y_true = np.asarray(y_true)
    probs  = np.asarray(probs)
    pred   = probs.argmax(axis=1)
    out = {
        'accuracy'       : accuracy_score(y_true, pred),
        'precision_macro': precision_score(y_true, pred, average='macro', zero_division=0),
        'recall_macro'   : recall_score(y_true, pred, average='macro', zero_division=0),
        'f1_macro'       : f1_score(y_true, pred, average='macro', zero_division=0),
        'f1_weighted'    : f1_score(y_true, pred, average='weighted', zero_division=0),
        'nll'            : log_loss(y_true, probs, labels=list(range(n_classes))),
    }
    try:
        out['auroc_macro_ovr'] = roc_auc_score(y_true, probs, multi_class='ovr', average='macro')
    except ValueError:
        out['auroc_macro_ovr'] = np.nan
    aps = []
    for c in range(n_classes):
        b = (y_true == c).astype(int)
        aps.append(average_precision_score(b, probs[:, c]) if b.sum() > 0 else np.nan)
    out['aupr_macro_ovr'] = float(np.nanmean(aps))
    return out

def align_proba(clf, p):
    """Expand predict_proba to all N_CLASSES columns when a fold's training
    split is missing a class, so probability matrices stay comparable."""
    full = np.zeros((p.shape[0], N_CLASSES), dtype=float)
    for j, c in enumerate(clf.classes_):
        full[:, int(c)] = p[:, j]
    s = full.sum(axis=1, keepdims=True)
    s[s == 0] = 1.0
    return full / s

## 6. Models and group-aware inner search

Each model is a pipeline whose first step is the per-fold `TruncatedSVD` on the genotype block; the ECFP block is passed through untouched. The inner CV is `StratifiedGroupKFold` on the outer-train rows, so tuning never sees a diplotype that the outer test fold holds.

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin

class GenotypeSVD(BaseEstimator, TransformerMixin):
    """TruncatedSVD on the first `n_gen` columns (genotype); ECFP passed through.

    Works in float64 and drops all-zero genotype columns before the
    decomposition. randomized_svd is numerically fragile on wide, very sparse
    float32 matrices and can emit non-finite values during its power
    iterations; both measures keep it stable here.
    """
    def __init__(self, n_gen, n_components=100, random_state=SEED):
        self.n_gen = n_gen
        self.n_components = n_components
        self.random_state = random_state

    def fit(self, X, y=None):
        G = np.ascontiguousarray(X[:, :self.n_gen], dtype=np.float64)
        G = np.nan_to_num(G, nan=0.0, posinf=0.0, neginf=0.0)
        self.keep_ = np.flatnonzero(G.any(axis=0))        # drop constant-zero positions
        if self.keep_.size == 0:
            self.keep_ = np.arange(min(2, G.shape[1]))
        G = G[:, self.keep_]
        cap = max(2, min(int(self.n_components), G.shape[0] - 1, G.shape[1] - 1))
        self.svd_ = TruncatedSVD(n_components=cap, algorithm='randomized',
                                 n_iter=7, random_state=self.random_state)
        self.svd_.fit(G)
        return self

    def transform(self, X):
        G = np.ascontiguousarray(X[:, :self.n_gen], dtype=np.float64)
        G = np.nan_to_num(G, nan=0.0, posinf=0.0, neginf=0.0)[:, self.keep_]
        Z = self.svd_.transform(G)
        out = np.hstack([Z, np.asarray(X[:, self.n_gen:], dtype=np.float64)])
        if not np.isfinite(out).all():
            raise FloatingPointError('GenotypeSVD produced non-finite output')
        return out

N_GEN = X_gen.shape[1]
X_all = np.hstack([X_gen, X_drug]).astype(np.float64)
print('combined feature matrix:', X_all.shape, X_all.dtype,
      '| finite:', bool(np.isfinite(X_all).all()))

# How many genotype columns actually vary at all across the 610 rows?
nz = int((X_gen.any(axis=0)).sum())
print(f'genotype columns that are not constant-zero: {nz} / {X_gen.shape[1]}')

def make_models():
    rf = Pipeline([
        ('svd', GenotypeSVD(N_GEN, n_components=SVD_MAX_COMPONENTS)),
        ('clf', RandomForestClassifier(random_state=SEED, n_jobs=-1)),
    ])
    rf_space = {
        'svd__n_components'    : randint(50, 300),
        'clf__n_estimators'    : randint(200, 800),
        'clf__max_depth'       : randint(5, 50),
        'clf__min_samples_split': randint(2, 10),
        'clf__min_samples_leaf': randint(1, 5),
        'clf__max_features'    : ['sqrt', 'log2', None],
    }

    svm = Pipeline([
        ('svd', GenotypeSVD(N_GEN, n_components=SVD_MAX_COMPONENTS)),
        ('sc',  StandardScaler()),
        ('clf', SVC(kernel='poly', probability=True, random_state=SEED)),
    ])
    svm_space = {
        'svd__n_components': randint(50, 300),
        'clf__C'           : loguniform(1e-2, 1e3),
        'clf__gamma'       : loguniform(1e-5, 1e0),
        'clf__degree'      : randint(2, 5)
    }

    xgb = Pipeline([
        ('svd', GenotypeSVD(N_GEN, n_components=SVD_MAX_COMPONENTS)),
        ('clf', XGBClassifier(objective='multi:softprob', num_class=N_CLASSES,
                              eval_metric='mlogloss', tree_method='hist',
                              random_state=SEED, n_jobs=-1)),
    ])
    xgb_space = {
        'svd__n_components'  : randint(50, 300),
        'clf__n_estimators'  : randint(200, 1000),
        'clf__learning_rate' : uniform(0.01, 0.25),
        'clf__max_depth'     : randint(3, 8),
        'clf__subsample'     : uniform(0.6, 0.4),
        'clf__colsample_bytree': uniform(0.6, 0.4),
    }

    return {
        'RandomForest': (rf,  rf_space),
        'SVM'         : (svm, svm_space),
        'XGBoost'     : (xgb, xgb_space),
    }

# Fallback parameters for SEARCH=False (smoke test only — never report these).
FIXED = {
    'RandomForest': {'svd__n_components':150, 'clf__n_estimators':515, 'clf__max_depth':13,
                     'clf__min_samples_split':5, 'clf__min_samples_leaf':2, 'clf__max_features':None},
    'SVM'         : {'svd__n_components':150, 'clf__C':0.0171, 'clf__gamma':0.0019,
                     'clf__degree':4},
    'XGBoost'     : {'svd__n_components':150, 'clf__n_estimators':847, 'clf__learning_rate':0.2567,
                     'clf__max_depth':3, 'clf__subsample':0.6022, 'clf__colsample_bytree':0.6298},
}

def inner_cv(y_tr, g_tr):
    """Group-aware inner CV; falls back to fewer splits if groups are too few."""
    for k in range(INNER_SPLITS, 1, -1):
        try:
            cv = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=SEED)
            list(cv.split(np.zeros(len(y_tr)), y_tr, g_tr))
            return cv
        except ValueError:
            continue
    raise RuntimeError('no usable inner split')

combined feature matrix: (622, 31334) float64 | finite: True
genotype columns that are not constant-zero: 10853 / 30310


In [7]:
# ---- informative-position diagnostics (Reviewer 3, limitation) ----
G3 = X_gen.reshape(len(data), TARGET_ROWS, TARGET_COLS)

# 1) across the 610 diplotype-drug rows
var_col = (X_gen.max(axis=0) != X_gen.min(axis=0))
var_pos = (G3.max(axis=0) != G3.min(axis=0)).any(axis=1)
print(f'[rows n={len(data)}]      variable channels: {var_col.sum():,} / {X_gen.shape[1]:,}'
      f' | variable positions: {var_pos.sum():,} / {TARGET_ROWS:,}')

# 2) across the unique diplotypes actually present
uniq_dip = data['DiplotypeGroup'].drop_duplicates().index.values
Gd = G3[uniq_dip]
vp_d = (Gd.max(axis=0) != Gd.min(axis=0)).any(axis=1)
print(f'[diplotypes n={len(uniq_dip)}] variable positions: {vp_d.sum():,} / {TARGET_ROWS:,}')

# 3) across all 361 single alleles in onehot_7 (the most fundamental number)
A = np.stack([onehot[k] for k in sorted(onehot)])
vp_a = (A.max(axis=0) != A.min(axis=0)).any(axis=1)
print(f'[alleles n={len(onehot)}]    variable positions: {vp_a.sum():,} / {TARGET_ROWS:,}')
print(f'                        → {100*vp_a.mean():.2f}% of the sequence carries any allele-level variation')

[rows n=622]      variable channels: 9,434 / 30,310 | variable positions: 4,330 / 4,330
[diplotypes n=104] variable positions: 4,330 / 4,330
[alleles n=361]    variable positions: 4,330 / 4,330
                        → 100.00% of the sequence carries any allele-level variation


## 7. Outer-fold loop

In [8]:
fold_rows = []
oof = {m: np.zeros((len(data), N_CLASSES)) for m in ['RandomForest','SVM','XGBoost']}
best_params_log = []

for fold in range(1, N_SPLITS + 1):
    te_idx = np.where(fold_id == fold)[0]
    tr_idx = np.where(fold_id != fold)[0]

    Xtr, ytr, gtr = X_all[tr_idx], y[tr_idx], groups[tr_idx]
    Xte, yte      = X_all[te_idx], y[te_idx]

    assert not (set(gtr) & set(groups[te_idx]))

    print('=' * 80)
    print(f'OUTER FOLD {fold}/{N_SPLITS}  train={len(tr_idx)} ({len(set(gtr))} diplotypes)  '
          f'test={len(te_idx)} ({len(set(groups[te_idx]))} diplotypes)')
    print('  test class counts:', np.bincount(yte, minlength=N_CLASSES).tolist())

    for name, (pipe, space) in make_models().items():
        if SEARCH:
            search = RandomizedSearchCV(
                pipe, space, n_iter=N_ITER, scoring='f1_macro',
                cv=inner_cv(ytr, gtr), n_jobs=-1, random_state=SEED,
                refit=True, verbose=0, error_score=np.nan,
            )
            search.fit(Xtr, ytr, groups=gtr)
            model  = search.best_estimator_          # the searched model IS the fitted model
            params = search.best_params_
            inner  = search.best_score_
        else:
            model  = pipe.set_params(**FIXED[name]).fit(Xtr, ytr)
            params = FIXED[name]
            inner  = np.nan

        proba = align_proba(model.named_steps['clf'], model.predict_proba(Xte))
        m = compute_metrics(yte, proba)

        oof[name][te_idx] = proba
        fold_rows.append({'model': name, 'fold': fold,
                          'n_train': len(tr_idx), 'n_test': len(te_idx),
                          'inner_cv_f1_macro': inner, **m})
        best_params_log.append({'model': name, 'fold': fold,
                                'params': json.dumps({k: str(v) for k, v in params.items()})})

        print(f'  {name:13s} acc {m["accuracy"]:.4f} | F1 {m["f1_macro"]:.4f} | '
              f'AUROC {m["auroc_macro_ovr"]:.4f} | AUPR {m["aupr_macro_ovr"]:.4f} | '
              f'NLL {m["nll"]:.4f}')

ml_folds = pd.DataFrame(fold_rows)
ml_folds.to_csv('ml_baselines_diplotype_grouped_fold_metrics.csv', index=False)
pd.DataFrame(best_params_log).to_csv('ml_baselines_selected_hyperparameters.csv', index=False)
print('\nsaved fold metrics and the hyperparameters actually used')

OUTER FOLD 1/5  train=530 (84 diplotypes)  test=92 (20 diplotypes)
  test class counts: [38, 31, 23]
  RandomForest  acc 0.4891 | F1 0.4422 | AUROC 0.7847 | AUPR 0.6340 | NLL 0.8159
  SVM           acc 0.4457 | F1 0.3199 | AUROC 0.7167 | AUPR 0.5494 | NLL 2.7305
  XGBoost       acc 0.5870 | F1 0.5659 | AUROC 0.8296 | AUPR 0.7117 | NLL 1.7250
OUTER FOLD 2/5  train=542 (88 diplotypes)  test=80 (16 diplotypes)
  test class counts: [59, 15, 6]
  RandomForest  acc 0.7000 | F1 0.6010 | AUROC 0.8005 | AUPR 0.7247 | NLL 0.7572
  SVM           acc 0.6625 | F1 0.5615 | AUROC 0.8128 | AUPR 0.6734 | NLL 0.9134
  XGBoost       acc 0.7000 | F1 0.5655 | AUROC 0.7982 | AUPR 0.6315 | NLL 0.9483
OUTER FOLD 3/5  train=392 (80 diplotypes)  test=230 (24 diplotypes)
  test class counts: [65, 80, 85]
  RandomForest  acc 0.5957 | F1 0.5768 | AUROC 0.8463 | AUPR 0.7174 | NLL 0.8597
  SVM           acc 0.4826 | F1 0.4781 | AUROC 0.7339 | AUPR 0.5706 | NLL 1.1628
  XGBoost       acc 0.7000 | F1 0.6844 | AUROC 0.

## 8. Out-of-fold predictions

Every row of `data_610.csv` gets a prediction from a model that never saw its diplotype. These feed the substrate-discordant analysis (Reviewer 2 point 6, Reviewer 3's direct test).

In [9]:
oof_df = data[['PharmGKB ID','Variant','DiplotypeGroup','Drug','Class','activity_score']].copy() \
         if 'activity_score' in data.columns else \
         data[['Variant','DiplotypeGroup','Drug','Class']].copy()
oof_df['fold']   = fold_id
oof_df['y_true'] = y

for name in ['RandomForest','SVM','XGBoost']:
    p = oof[name]
    oof_df[f'{name}_pred'] = p.argmax(axis=1)
    for c, cname in class_mapping.items():
        oof_df[f'{name}_prob_{cname}'] = p[:, c]

oof_df.to_csv('ml_baselines_diplotype_grouped_oof_predictions.csv', index=False)

print('POOLED OUT-OF-FOLD PERFORMANCE')
pooled_rows = []
for name in ['RandomForest','SVM','XGBoost']:
    pm = compute_metrics(y, oof[name])
    pooled_rows.append({'model': name, **pm})
    print(f'  {name:13s} acc {pm["accuracy"]:.4f} | F1 {pm["f1_macro"]:.4f} | '
          f'AUROC {pm["auroc_macro_ovr"]:.4f} | AUPR {pm["aupr_macro_ovr"]:.4f} | NLL {pm["nll"]:.4f}')
pooled_ml = pd.DataFrame(pooled_rows)

POOLED OUT-OF-FOLD PERFORMANCE
  RandomForest  acc 0.5804 | F1 0.5573 | AUROC 0.8070 | AUPR 0.6790 | NLL 0.8259
  SVM           acc 0.5434 | F1 0.5120 | AUROC 0.7440 | AUPR 0.5765 | NLL 1.4106
  XGBoost       acc 0.6576 | F1 0.6580 | AUROC 0.8518 | AUPR 0.7521 | NLL 1.0803


## 9. Table 1: every model, one protocol

Pulls in the deep-model fold metrics if their CSVs are in the folder, so the summary covers all five rows. Both aggregations are reported, because with fold sizes from 22 to 139 the unweighted fold mean and the pooled out-of-fold estimate can disagree — and which one you quote should not be a choice made after seeing them.

In [12]:
DEEP_FILES = {
    'Single CNN': 'genotype_only_diplotype_grouped_5fold_metrics.csv',
    'Multimodal': 'multimodal_diplotype_grouped_5fold_metrics.csv',
}

frames = [ml_folds]
for name, path in DEEP_FILES.items():
    if os.path.exists(path):
        d = pd.read_csv(path)
        d['model'] = name
        frames.append(d)
        print('loaded', path)
    else:
        print('[missing]', path, '- deep rows will be absent from the table')

allf = pd.concat(frames, ignore_index=True, sort=False)
allf.to_csv('table1_grouped_cv_all_fold_metrics.csv', index=False)

METRICS = ['accuracy','precision_macro','recall_macro','f1_macro',
           'auroc_macro_ovr','aupr_macro_ovr','nll']
ORDER = ['RandomForest','SVM','XGBoost','Single CNN','Multimodal']
present = [m for m in ORDER if m in set(allf.model)]

rows = []
for mdl in present:
    s = allf[allf.model == mdl]
    r = {'model': mdl}
    for k in METRICS:
        v = s[k].dropna()
        r[k] = f'{v.mean():.4f} ± {v.std(ddof=1):.4f}' + ('' if len(v) == N_SPLITS else f' (n={len(v)})')
    # sample-weighted fold mean, for contrast with the unweighted one
    w = s['n_test'].values if 'n_test' in s else np.ones(len(s))
    r['accuracy_weighted'] = f"{np.average(s['accuracy'].values, weights=w):.4f}"
    rows.append(r)

summary = pd.DataFrame(rows)
summary.to_csv('table1_grouped_cv_summary.csv', index=False)
print('\n=== MEAN ± SD ACROSS OUTER FOLDS (diplotype-grouped) ===')
display(summary)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
loaded genotype_only_diplotype_grouped_5fold_metrics.csv
loaded multimodal_diplotype_grouped_5fold_metrics.csv

=== MEAN ± SD ACROSS OUTER FOLDS (diplotype-grouped) ===


,model,accuracy,precision_macro,recall_macro,f1_macro,auroc_macro_ovr,aupr_macro_ovr,nll,accuracy_weighted
0,RandomForest,0.5857 ± 0.0883,0.6089 ± 0.1007,0.5474 ± 0.0576,0.5360 ± 0.0621,0.8167 ± 0.0279,0.7025 ± 0.0475,0.8102 ± 0.0519,0.5804
1,SVM,0.5613 ± 0.0932,0.5237 ± 0.1366,0.5062 ± 0.1067,0.4732 ± 0.1126,0.7717 ± 0.0525,0.6160 ± 0.0797,1.4940 ± 0.7216,0.5434
2,XGBoost,0.6527 ± 0.0617,0.6684 ± 0.0806,0.6161 ± 0.0597,0.5961 ± 0.0571,0.8533 ± 0.0405,0.7390 ± 0.0698,1.1103 ± 0.3583,0.6576
3,Single CNN,0.5331 ± 0.1714,0.4814 ± 0.0896,0.4544 ± 0.1335,0.4005 ± 0.1532,0.7390 ± 0.1114,0.6020 ± 0.0770,0.9564 ± 0.2676,0.4791
4,Multimodal,0.6522 ± 0.1133,0.6712 ± 0.0444,0.6216 ± 0.0896,0.6214 ± 0.0856,0.8400 ± 0.0260,0.7141 ± 0.0320,1.0391 ± 0.2676,0.6093


## 10. Paired significance tests

Folds are paired across models because every model saw the identical split. Reviewer 2 asked for p-values on every row; Reviewer 3 noted a paired comparison costs nothing given ten trained models.

In [11]:
REFERENCE = 'Multimodal'   # change to compare against a different row

test_rows = []
if REFERENCE in present:
    ref = allf[allf.model == REFERENCE].set_index('fold')
    for mdl in present:
        if mdl == REFERENCE:
            continue
        oth = allf[allf.model == mdl].set_index('fold')
        common = sorted(set(ref.index) & set(oth.index))
        for k in METRICS:
            a = ref.loc[common, k].values.astype(float)   # reference
            b = oth.loc[common, k].values.astype(float)   # comparator
            ok = ~(np.isnan(a) | np.isnan(b))
            a, b = a[ok], b[ok]
            if len(a) < 3:
                continue
            d = b - a
            try:
                pw = wilcoxon(b, a).pvalue
            except ValueError:
                pw = np.nan
            test_rows.append({
                'reference': REFERENCE, 'comparator': mdl, 'metric': k, 'n_folds': len(a),
                'ref_mean': a.mean(), 'cmp_mean': b.mean(), 'diff_cmp_minus_ref': d.mean(),
                'p_paired_t': ttest_rel(b, a).pvalue, 'p_wilcoxon': pw,
                'cmp_wins': int((d > 0).sum()),
            })

paired = pd.DataFrame(test_rows)
if len(paired):
    paired.to_csv('table1_grouped_cv_paired_tests.csv', index=False)
    display(paired.round(4))
else:
    print('No deep-model fold metrics found, so no paired tests were run.')

,reference,comparator,metric,n_folds,ref_mean,cmp_mean,diff_cmp_minus_ref,p_paired_t,p_wilcoxon,cmp_wins
0,Multimodal,RandomForest,accuracy,5,0.6209,0.5857,-0.0351,0.7204,0.8125,3
1,Multimodal,RandomForest,precision_macro,5,0.6029,0.6089,0.0060,0.9617,1.0000,2
2,Multimodal,RandomForest,recall_macro,5,0.5388,0.5474,0.0087,0.9222,1.0000,2
3,Multimodal,RandomForest,f1_macro,5,0.5216,0.5360,0.0145,0.8736,0.8125,3
4,Multimodal,RandomForest,auroc_macro_ovr,5,0.8231,0.8167,-0.0064,0.9200,1.0000,2
5,Multimodal,RandomForest,aupr_macro_ovr,5,0.7012,0.7025,0.0013,0.9898,0.8125,2
6,Multimodal,RandomForest,nll,5,1.3323,0.8102,-0.5221,0.3074,0.6250,2
7,Multimodal,SVM,accuracy,5,0.6209,0.5613,-0.0595,0.5149,1.0000,3
8,Multimodal,SVM,precision_macro,5,0.6029,0.5237,-0.0792,0.5991,0.6250,2
9,Multimodal,SVM,recall_macro,5,0.5388,0.5062,-0.0326,0.7891,0.8125,2


## 11. Pooled out-of-fold confusion matrices

In [12]:
for name in ['RandomForest','SVM','XGBoost']:
    pred = oof[name].argmax(axis=1)
    print('=' * 60)
    print(name)
    print(confusion_matrix(y, pred, labels=list(range(N_CLASSES))))
    print(classification_report(y, pred,
                                target_names=[class_mapping[i] for i in range(N_CLASSES)],
                                digits=4, zero_division=0))

RandomForest
[[193  33  14]
 [ 51 140   4]
 [105   8  62]]
              precision    recall  f1-score   support

          de     0.5530    0.8042    0.6553       240
           f     0.7735    0.7179    0.7447       195
         nor     0.7750    0.3543    0.4863       175

    accuracy                         0.6475       610
   macro avg     0.7005    0.6255    0.6288       610
weighted avg     0.6872    0.6475    0.6354       610

SVM
[[199  30  11]
 [ 36 157   2]
 [ 59   3 113]]
              precision    recall  f1-score   support

          de     0.6769    0.8292    0.7453       240
           f     0.8263    0.8051    0.8156       195
         nor     0.8968    0.6457    0.7508       175

    accuracy                         0.7689       610
   macro avg     0.8000    0.7600    0.7706       610
weighted avg     0.7877    0.7689    0.7694       610

XGBoost
[[173  51  16]
 [ 43 149   3]
 [ 82   9  84]]
              precision    recall  f1-score   support

          de     0.5

## 12. How to read and report this

**One protocol.** Every row now comes from the same diplotype-disjoint folds, so
Table 1 is internally comparable and answers Reviewer 1's and Reviewer 3's request
for group-aware validation as the primary evaluation. State in Methods that the
previous sample-level split is retained only as a secondary analysis, and say
plainly that the grouped numbers are lower because the earlier split let the same
star-allele family appear on both sides.

**Report both aggregations.** `accuracy` (unweighted fold mean ± SD) and
`accuracy_weighted` can point in different directions here, since fold sizes run
from 22 to 139. Print both. If they disagree, that disagreement *is* the finding —
it means the models are not separated by this data, and choosing the flattering
one after the fact is the thing a reviewer with your code will catch.

**One asymmetry stays.** The ML baselines get a hyperparameter search inside every
fold; the deep models used a learning rate fixed a priori. That favours the
baselines. Running the deep hyperparameter search per fold is the clean fix; if
that is out of scope, say so in Methods rather than leaving it unstated.

**Calibration.** `nll` is in the table for every model. If a model wins on accuracy
but loses badly on NLL, say that — it matters more than a two-decimal accuracy gap
for anything clinical.

**Next.** `ml_baselines_diplotype_grouped_oof_predictions.csv` gives every sample a
prediction from a model blind to its diplotype. Joined with the deep models' OOF
files, it supports the substrate-discordant analysis: hold the diplotype fixed,
vary the drug across pairs whose true phenotype differs, and ask whether each
model's prediction follows the drug. That is the test Reviewer 3 made a condition
of publication, and it is where a drug-aware model can show something a
genotype-only model structurally cannot.
